# 04 · 전체 볼륨 loss 분포와 슬라이스 오류 분석

새 Linux GPU 서버에서 **기존 모델의 순전파만 실행**합니다. 재학습·데이터 다운로드·재전처리 준비 작업은 필요하지 않습니다. 추론 중 모델에 저장된 전처리 설정을 자동 적용합니다.

`nnUNet/`과 복사한 `acdc-data/`를 같은 상위 폴더에 두세요. 예: `/root/nnUNet`, `/root/acdc-data`. 기존 서버의 절대 경로가 기록된 config.server.json은 사용하지 않습니다.

필수 복사 항목: `acdc-data/nnUNet_raw/Dataset027_ACDC/{imagesTr,labelsTr}` 및 `acdc-data/runs/acdc-baseline-v1/`의 config.snapshot.json, splits_final.json, 모델 폴더(plans.json, dataset.json, fold_0/checkpoint_final.safetensors). 기존 acdc-data 전체를 복사해도 됩니다.

1. 기존 Python 3.11 venv의 Jupyter 커널 선택 → 경로 초기화 → 필요 시 패키지 설치.
2. `RUN_INFERENCE=True`로 설정 후 GPU 서버에서 200개 확률 출력 생성. 완료 사례는 재실행 시 건너뜁니다.
3. CPU에서 loss 계산·분포 분석·환자별 슬라이스 탐색. 생성 결과까지 복사하면 로컬에서도 3단계만 실행할 수 있습니다.

**범위:** ACDC 공개 학습 세트 100명·ED/ES 200개. 선택한 한 fold의 train 160개 / validation 40개를 같은 모델로 비교합니다. 숨겨진 정답의 공식 test 세트나 4D 전체 심장 주기는 포함하지 않습니다.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, platform, shutil
sys.dont_write_bytecode = True
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'nnunetv2').is_dir()), None)
if REPO is None:
    raise RuntimeError('이 저장소 안에서 노트북을 여세요.')
STORAGE_ROOT = (REPO.parent / 'acdc-data').resolve()  # 필요하면 절대 경로로 변경
RUN_ID = 'acdc-baseline-v1'
FOLD = 0
CHECKPOINT = 'checkpoint_final.safetensors'
GPU = '0'  # 새 서버에서 사용할 GPU 번호
INSTALL_PACKAGES = False  # 새 서버의 venv에서 처음 한 번 True
RUN_INFERENCE = False  # GPU 서버에서 확률 출력을 생성할 때만 True

os.environ['CUDA_VISIBLE_DEVICES'] = GPU
os.environ['nnUNet_compile'] = 'false'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['nnUNet_raw'] = str(STORAGE_ROOT / 'nnUNet_raw')
os.environ['nnUNet_preprocessed'] = str(STORAGE_ROOT / 'nnUNet_preprocessed')
os.environ['nnUNet_results'] = str(STORAGE_ROOT / 'runs' / RUN_ID / 'nnUNet_results')
SCRIPTS = REPO / 'experiments/acdc/scripts'
sys.path.insert(0, str(SCRIPTS))
print('Kernel:', sys.executable)
print('Repo:', REPO)
print('Data:', STORAGE_ROOT)
if not STORAGE_ROOT.is_dir():
    raise FileNotFoundError('acdc-data를 저장소 옆에 복사하거나 STORAGE_ROOT를 수정하세요.')
print('Free disk GiB:', round(shutil.disk_usage(STORAGE_ROOT).free / 2**30, 2))

## 초기 환경 설치 (새 서버에서 한 번)
기존 Jupyter 서버 자체를 다시 설치하지 않습니다. **현재 선택한 커널의 venv**에 설치합니다. Python 3.11을 권장합니다. 먼저 터미널에서 `python -m pip install ipykernel`과 `python -m ipykernel install --user --name nnunet311 --display-name "Python 3.11 (nnUNet)"`를 해당 venv로 실행하고 커널을 선택하세요.

아래 설치에는 PyTorch 등 큰 패키지가 포함되므로 GPU 서버에서만 실행합니다. 설치 후 커널을 재시작하고 첫 셀의 INSTALL_PACKAGES를 False로 되돌리세요. 서버 이미지에 CUDA용 PyTorch가 있으면 기존 설치를 활용합니다. CUDA 확인이 실패하면 드라이버/서버에 맞는 PyTorch 설치가 필요합니다. 원래 환경 기록은 `acdc-data/environment.lock.txt`에서 참고할 수 있습니다. 다른 환경에서의 추론 결과가 과거 validation 결과와 완전히 같다고 가정하지 않습니다.

In [ ]:
if INSTALL_PACKAGES:
    if platform.system() != 'Linux' or sys.prefix == sys.base_prefix:
        raise RuntimeError('새 Linux 서버의 venv 커널에서만 설치하세요.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
                           '-e', str(REPO), 'matplotlib>=3.8,<4', 'pandas>=2,<3',
                           'SimpleITK>=2.2', 'ipywidgets>=8,<9'])
    print('설치 완료. 커널 재시작 후 INSTALL_PACKAGES=False로 설정하세요.')
else:
    print('설치 생략. 아래 CPU 분석에는 numpy/pandas/matplotlib/SimpleITK가 필요합니다.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import importlib
import loss_analysis
importlib.reload(loss_analysis)
from loss_analysis import context, analyse, load_case, show_slice
ctx = context(STORAGE_ROOT, RUN_ID, FOLD, CHECKPOINT)
print('Model:', ctx['model'])
print('New outputs:', ctx['out'])
print(pd.Series([s for s, _ in ctx['cases']]).value_counts().to_string())
print('완료 사례:', sum((ctx['out'] / s / f'{c}.complete.json').exists() for s, c in ctx['cases']))

## 1. 순전파로 마스크와 클래스별 확률 생성
선택한 fold의 **동일한 checkpoint**로 train/validation 모두 다시 추론합니다. sliding window(step 0.5), Gaussian 합성, mirroring을 사용합니다. 원래 좌표계로 복원한 확률 `[class,z,y,x]`를 `.npz`로 저장합니다. 기존 validation과 학습 결과는 덮어쓰지 않습니다.

새 결과는 `runs/<run_id>/loss_analysis/fold_0/checkpoint_final/{train,val}`에 저장됩니다. 확률 파일은 마스크보다 훨씬 큽니다. 실행 전 원본 voxel 수 기반 여유 용량 검사를 수행하며, 실제 압축률/임시 메모리에 따라 필요한 양은 달라집니다. 데이터셋은 복제하지 않습니다.

별도 Python 프로세스로 실행하므로 Jupyter multiprocessing 문제를 피합니다. 로그는 새 결과 상위 폴더의 checkpoint_final-inference.log에 남습니다. 셀 중단 시 자식 프로세스도 종료하고, 재실행하면 완료 표시가 없는 사례부터 처리합니다. 동시에 같은 출력 경로에서 실행하지 마세요. 완료 결과와 다른 모델·코드·환경을 섞지 않도록 manifest를 확인합니다.

In [ ]:
if RUN_INFERENCE:
    if platform.system() != 'Linux':
        raise RuntimeError('이 컴퓨터에서는 추론하지 마세요. Linux GPU 서버에서 실행하세요.')
    import torch
    import nnunetv2
    if Path(nnunetv2.__file__).resolve().parent != REPO / 'nnunetv2':
        raise RuntimeError('현재 커널에 이 저장소를 pip install -e 로 설치하세요.')
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA 사용 불가. CUDA용 PyTorch와 NVIDIA 드라이버를 확인하세요.')
    print('GPU:', torch.cuda.get_device_name(0))
    # Keep the log outside the manifest-managed output directory until initialization.
    log_dir = ctx['out'].parent
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / (ctx['out'].name + '-inference.log')
    cmd = [sys.executable, '-u', str(SCRIPTS / 'loss_analysis.py'),
           '--storage', str(STORAGE_ROOT), '--run-id', RUN_ID,
           '--fold', str(FOLD), '--checkpoint', CHECKPOINT]
    with log_path.open('a', buffering=1) as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1, env=os.environ.copy())
        try:
            for line in proc.stdout:
                log.write(line)
                print(line, end='')
            if proc.wait() != 0:
                raise RuntimeError(f'추론 실패. 로그 확인: {log_path}')
        finally:
            if proc.poll() is None:
                proc.terminate()
                try:
                    proc.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    proc.kill()
                    proc.wait()
    print('완료:', log_path)
else:
    print('추론 생략. 확률 파일이 없으면 서버에서 RUN_INFERENCE=True로 실행하세요.')

## 2. Loss 정의와 전체 데이터 집계
**아래 값은 학습 로그의 loss를 재현한 값이 아니라, 저장된 전체 볼륨 확률의 평가 loss입니다.** 학습에서는 증강된 패치, batch Dice, 여러 해상도의 deep supervision을 사용합니다. 여기서는 증강 없이 복원된 원본 voxel 전체와 추론 시 mirroring을 사용합니다.

- `ce`: 모든 voxel의 `−log(p[정답 클래스])` 평균. 로그 계산 시 확률을 `[1e-7,1]`로 제한합니다.
- `soft_dice`: 배경을 제외한 RV/MLV/LVC의 `(2Σpy + 1e-5)/(Σp + Σy + 1e-5)` 평균.
- **`eval_loss = ce − soft_dice`**: nnU-Net의 음수 Dice 부호를 따른 볼륨 평가용 값. 낮을수록 좋으며 음수가 정상입니다.
- `soft_dice_loss = 1 − soft_dice`: 0에 가까울수록 좋습니다. hard Dice는 argmax 마스크로 계산하며 양쪽 모두 없는 클래스는 제외합니다.
- `foreground_ce`: GT foreground에서만 계산한 CE. 전체 CE가 많은 배경에 의해 작아지는 현상을 함께 확인합니다.

슬라이스 soft Dice는 해당 슬라이스의 세 클래스를 모두 포함하므로, 해부학적 구조의 존재 여부에도 영향을 받습니다. 슬라이스 loss 평균은 볼륨 loss와 같지 않습니다. 형상 불일치·잘못된 확률은 자동 보정 없이 오류로 처리합니다. 모든 사례가 완료되지 않으면 전체 데이터 분포를 그리지 않습니다.

In [ ]:
cases, slices = analyse(ctx)  # CPU; 한 번에 한 볼륨만 읽음
print(f'{len(cases)} volumes / {cases.patient.nunique()} patients / {len(slices)} slices')
display(cases.groupby('subset')[['eval_loss', 'ce', 'soft_dice_loss', 'foreground_ce', 'hard_dice']]
        .agg(['count', 'mean', 'median', 'std', 'min', 'max']))
print('CSV:', ctx['out'] / 'reports')

## 3. Train / validation loss histogram
두 집단의 크기가 다르므로 공통 bin과 **집단 내 비율(%)**을 사용합니다. 각 관측치는 3D 볼륨 한 개입니다. ED/ES는 같은 환자이므로 독립 표본으로 가정하지 않습니다. Train 분포는 학습에 사용된 사례의 성능이며 일반화 성능이 아닙니다.

In [ ]:
metrics = ['eval_loss', 'ce', 'soft_dice_loss', 'foreground_ce']
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for ax, metric in zip(axes.flat, metrics):
    values = cases[metric].to_numpy()
    bins = np.histogram_bin_edges(values[np.isfinite(values)], bins=20)
    for subset, color in [('train', 'tab:blue'), ('val', 'tab:orange')]:
        vals = cases.loc[cases.subset == subset, metric].dropna().to_numpy()
        ax.hist(vals, bins=bins, weights=np.full(len(vals), 100 / len(vals)),
                alpha=.45, label=f'{subset} (n={len(vals)})', color=color)
    ax.set(xlabel=metric, ylabel='Volumes within subset (%)')
    ax.legend()
plt.show()
plt.close(fig)
# Patient-level average reduces double counting of ED/ES when inspecting the gap.
patient_losses = cases.groupby(['subset', 'patient'])[metrics].mean()
display(patient_losses.groupby('subset').agg(['count', 'mean', 'median', 'std']))

## 4. 높은 loss 사례·클래스·슬라이스를 체계적으로 찾기
각 집단의 상위 10%를 탐색 후보로 표시합니다. 이 기준은 오류 판정이나 통계적 유의성 검정이 아닙니다. 높은 CE가 특정 클래스, 작은 foreground 비율, 일부 슬라이스에 집중되는지 비교하세요. 낮은 점수만 선택해서 전체 성능을 대표하는 것으로 해석하지 마세요.

In [ ]:
ranked = cases.copy()
ranked['subset_p90'] = ranked.groupby('subset').eval_loss.transform(lambda x: x.quantile(.9))
ranked['high_loss'] = ranked.eval_loss >= ranked.subset_p90
cols = ['subset', 'case', 'eval_loss', 'ce', 'foreground_ce', 'hard_dice', 'foreground_fraction']
for subset in ['train', 'val']:
    print(subset, 'highest loss')
    display(ranked[ranked.subset == subset].nlargest(10, 'eval_loss')[cols])
display(cases.groupby('subset')[[f'ce_{r}' for r in ['RV', 'MLV', 'LVC']] +
                               [f'hard_dice_{r}' for r in ['RV', 'MLV', 'LVC']]].mean())
fig, ax = plt.subplots(figsize=(7, 4))
for subset in ['train', 'val']:
    part = cases[cases.subset == subset]
    ax.scatter(part.foreground_fraction, part.eval_loss, alpha=.65, label=subset)
ax.set(xlabel='GT foreground voxel fraction', ylabel='Whole-volume evaluation loss')
ax.legend()
plt.show()
plt.close(fig)
ranked.sort_values('eval_loss', ascending=False).to_csv(ctx['out'] / 'reports/ranked-cases.csv', index=False)

## 5. 환자 번호와 ED/ES frame 선택
`PATIENT_NUMBER=34`처럼 1~100을 입력하세요. `FRAME_NUMBER=None`이면 해당 환자에서 loss가 가장 높은 frame을 선택합니다. 표의 `frame01`, `frame16` 등의 **숫자**를 입력하면 직접 선택합니다. Frame 번호는 원본 ED/ES 시점 번호이며 z 슬라이스 번호와 다릅니다.

여기서 보여주는 것은 선택한 ED/ES의 **3D 영상을 z 방향으로 자른 2D 슬라이스**입니다. 전체 4D 시간축 동영상은 아닙니다.

In [ ]:
PATIENT_NUMBER = 34
FRAME_NUMBER = None  # 예: 16
patient = f'patient{int(PATIENT_NUMBER):03d}'
options = cases[cases.patient == patient].sort_values('frame')
if options.empty:
    raise ValueError(f'환자를 찾을 수 없습니다: {patient}')
display(options[cols + ['frame', 'slices']])
if FRAME_NUMBER is None:
    selected = options.loc[options.eval_loss.idxmax()]
else:
    match = options[options.frame == f'frame{int(FRAME_NUMBER):02d}']
    if match.empty:
        raise ValueError('위 표에 있는 frame 번호를 선택하세요.')
    selected = match.iloc[0]
subset, case = selected['subset'], selected['case']
loaded = load_case(ctx, subset, case, with_image=True)
case_slices = slices[(slices.case == case) & (slices.subset == subset)].sort_values('z')
display(case_slices[['z', 'eval_loss', 'ce', 'foreground_ce', 'hard_dice', 'error_fraction']])
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(case_slices.z, case_slices.ce, 'o-', label='all-voxel CE')
axes[0].plot(case_slices.z, case_slices.foreground_ce, 'o-', label='foreground CE')
axes[1].plot(case_slices.z, case_slices.hard_dice, 'o-', label='hard Dice')
for ax in axes:
    ax.set_xlabel('z slice (0-based)')
    ax.legend()
fig.suptitle(f'{case} ({subset})')
plt.show()
plt.close(fig)

## 6. 슬라이스 탐색 및 전체 슬라이스 출력
슬라이더로 한 장씩 이동합니다. 위젯이 지원되지 않는 Jupyter에서는 다음 셀의 `SLICE_INDEX`를 수정해 사용하세요. 정답과 예측의 클래스 색은 RV=초록, MLV=주황, LVC=파랑입니다. 오류는 FP=노랑, FN=빨강, foreground 클래스 혼동=보라입니다.

물리적 픽셀 간격을 반영하고 원본 배열 방향을 유지합니다. 별도 재정렬·리샘플링하지 않으며 방사선학적 좌우 표기를 의미하지 않습니다. CE 색상 범위는 해당 볼륨 P99.5로 고정되며, 표시 범위 밖 값도 실제 loss에는 포함됩니다.

In [ ]:
worst_z = int(case_slices.loc[case_slices.ce.idxmax(), 'z'])
try:
    import ipywidgets as widgets
except ImportError:
    print('ipywidgets 없음: 다음 셀의 SLICE_INDEX를 사용하세요.')
    show_slice(ctx, subset, case, worst_z, loaded=loaded)
else:
    slider = widgets.IntSlider(value=worst_z, min=0, max=loaded[0].shape[0]-1,
                               description='z slice', continuous_update=False)
    viewer = widgets.interactive_output(
        lambda z: show_slice(ctx, subset, case, z, loaded=loaded), {'z': slider})
    display(slider, viewer)

In [ ]:
SLICE_INDEX = worst_z  # 0부터 시작
SHOW_ALL_SLICES = False  # True면 이 3D 볼륨의 모든 슬라이스를 순서대로 출력
for z in range(loaded[0].shape[0]) if SHOW_ALL_SLICES else [SLICE_INDEX]:
    show_slice(ctx, subset, case, z, loaded=loaded)

## 해석과 다음 실험
- train/validation 모두 높은 loss: 공통으로 어려운 영상·클래스·전처리·정답 품질을 먼저 살펴봅니다.
- validation 쪽의 높은 loss: 일반화 차이의 단서입니다. 단일 fold·20명이라는 한계를 고려하고 다른 fold로 확인하세요.
- CE만 크고 Dice는 높은 경우: 일부 voxel에 대한 과도한 확신 등 확률 품질을 확인합니다. Dice만 낮으면 구조의 크기·경계·누락을 함께 확인합니다.
- 특정 z에서만 악화: 실제 해부학적 위치를 시각적으로 확인하세요. z 인덱스만으로 base/apex를 단정하지 않습니다.
- 이 분석을 보고 구조/하이퍼파라미터를 선택하면 validation을 개발에 사용한 것입니다. 최종 주장은 별도 평가 또는 적절한 교차검증으로 확인하세요.

산출물: 새 확률/마스크/완료 표시, 추론 manifest(모델·입력·코드·패키지 해시), 추론 로그, case-losses.csv, slice-losses.csv, ranked-cases.csv. 그래프는 노트북에서만 표시하며 대용량 결과를 저장소 docs에 복사하지 않습니다.